# DE-02 — Ingestion & Incremental Processing

**Dataset:** `data/loan_data_02.csv`

This notebook demonstrates full load, append, upsert, snapshot, watermark, CDC, late-arriving data, replay, deletion handling, schema evolution, throttling, and extraction consistency.

> The source CSV has no event timestamp. For training only, this notebook adds a deterministic synthetic `Event_TS` based on row order. The original CSV remains unchanged.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_02.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

## Learning Content

### Load patterns

- **Full load:** replace the target with a complete source copy.
- **Append:** add new records; safe only when arrivals are unique and immutable.
- **Upsert:** insert new business keys and update existing keys.
- **Snapshot:** retain the complete observed state at a point in time.

### Incremental state

A **watermark/high-water mark** records the largest successfully published event value. CDC represents changes as inserts, updates, and deletes. A pipeline must advance its checkpoint only after successful publication.

### Operational edge cases

- Late data requires an overlap window or event-time strategy.
- Replay/backfill must be bounded and idempotent.
- Source deletion requires tombstones or snapshot comparison.
- Schema evolution requires explicit compatibility rules.
- Throttling protects the source.
- Extraction consistency detects source changes during a read.

In [ ]:
from hashlib import sha256

source = raw.copy()
source["Event_TS"] = (
    pd.Timestamp("2026-01-01 00:00:00", tz="UTC")
    + pd.to_timedelta(np.arange(len(source)), unit="h")
)

def dataframe_fingerprint(frame: pd.DataFrame) -> str:
    stable = frame.sort_values("Loan_ID").to_csv(index=False).encode("utf-8")
    return sha256(stable).hexdigest()

before_hash = dataframe_fingerprint(source)
extracted = source.copy()
after_hash = dataframe_fingerprint(source)
assert before_hash == after_hash, "Source changed during extraction"

print("Extraction fingerprint:", before_hash[:16])
print(extracted[["Loan_ID", "Event_TS"]].head().to_string(index=False))

## Hands-on / Demonstration

### Full, append, upsert, and snapshot

In [ ]:
business_key = "Loan_ID"

# Full load
target = extracted.iloc[:20].copy()

# Append only rows whose business keys do not exist.
incoming_append = extracted.iloc[20:25].copy()
new_rows = incoming_append.loc[~incoming_append[business_key].isin(target[business_key])]
target = pd.concat([target, new_rows], ignore_index=True)

# Upsert: one existing row is changed and one new row is supplied.
incoming_upsert = extracted.iloc[[2, 25]].copy()
incoming_upsert.loc[incoming_upsert.index[0], "LoanAmount"] += 10
target = (
    pd.concat([target, incoming_upsert], ignore_index=True)
    .drop_duplicates(subset=[business_key], keep="last")
)

# Snapshot retains observation metadata.
snapshot = extracted.copy()
snapshot["snapshot_id"] = "snapshot-2026-01-01"
snapshot["snapshot_at"] = pd.Timestamp.now(tz="UTC")

assert target[business_key].is_unique
print("Target rows after append/upsert:", len(target))
print("Snapshot rows:", len(snapshot))

### Timestamp watermark extraction and late arrivals

An overlap window intentionally rereads recent records. The upsert key makes that reread safe.

In [ ]:
watermark = source["Event_TS"].iloc[29]
overlap = pd.Timedelta(hours=2)
incremental_batch = source.loc[source["Event_TS"] > watermark - overlap].copy()

# Simulate a late record whose event time falls inside the overlap.
late_record = source.iloc[[5]].copy()
late_record["Loan_ID"] = "LP_LATE_001"
late_record["Event_TS"] = watermark - pd.Timedelta(hours=1)
incremental_batch = pd.concat([incremental_batch, late_record], ignore_index=True)

published = source.loc[source["Event_TS"] <= watermark].copy()
published = (
    pd.concat([published, incremental_batch], ignore_index=True)
    .drop_duplicates("Loan_ID", keep="last")
)

assert "LP_LATE_001" in set(published["Loan_ID"])
assert published["Loan_ID"].is_unique
new_watermark = incremental_batch["Event_TS"].max()
print("Previous watermark:", watermark)
print("New watermark:", new_watermark)
print("Published rows:", len(published))

### CDC, source deletion, replay, and schema evolution

In [ ]:
previous_snapshot = source.iloc[:30].copy()
current_snapshot = previous_snapshot.copy()

# Simulate one update, one deletion, and one insertion.
current_snapshot.loc[current_snapshot.index[1], "LoanAmount"] += 25
deleted_id = current_snapshot.iloc[2]["Loan_ID"]
current_snapshot = current_snapshot[current_snapshot["Loan_ID"] != deleted_id]
inserted = source.iloc[[30]].copy()
current_snapshot = pd.concat([current_snapshot, inserted], ignore_index=True)

previous_by_key = previous_snapshot.set_index("Loan_ID")
current_by_key = current_snapshot.set_index("Loan_ID")
insert_ids = current_by_key.index.difference(previous_by_key.index)
delete_ids = previous_by_key.index.difference(current_by_key.index)
common_ids = current_by_key.index.intersection(previous_by_key.index)
update_ids = [
    key for key in common_ids
    if not current_by_key.loc[key].equals(previous_by_key.loc[key])
]

cdc_events = pd.DataFrame(
    [(key, "INSERT") for key in insert_ids]
    + [(key, "UPDATE") for key in update_ids]
    + [(key, "DELETE") for key in delete_ids],
    columns=["Loan_ID", "operation"],
)

# Backward-compatible schema evolution: optional column with a default.
evolved = current_snapshot.copy()
evolved["Source_System"] = "training_csv"

print(cdc_events.to_string(index=False))
print("Evolved column:", evolved["Source_System"].unique().tolist())

### Rerun without duplicates and source throttling

The same batch is applied twice. Deduplication on the business key makes the result stable. A production extractor would sleep between chunks; this demonstration records chunks without adding delay.

In [ ]:
def idempotent_upsert(existing: pd.DataFrame, incoming: pd.DataFrame) -> pd.DataFrame:
    return (
        pd.concat([existing, incoming], ignore_index=True)
        .drop_duplicates("Loan_ID", keep="last")
        .sort_values("Loan_ID")
        .reset_index(drop=True)
    )

batch = source.iloc[10:20].copy()
once = idempotent_upsert(source.iloc[:15].copy(), batch)
twice = idempotent_upsert(once, batch)
assert dataframe_fingerprint(once) == dataframe_fingerprint(twice)

chunk_size = 10
chunks = [source.iloc[start:start + chunk_size] for start in range(0, len(source), chunk_size)]
print("Controlled extraction chunk sizes:", [len(chunk) for chunk in chunks])
print("Idempotent rerun verified:", len(twice), "unique rows")

## Enterprise Control

Every production pipeline requires a restart/replay strategy and source-impact control.

Required controls:

- Persist the checkpoint only after the target commit.
- Use a deterministic idempotency key.
- Bound replay by start/end values.
- Record inserts, updates, and deletes.
- Limit extraction concurrency, chunk size, and request rate.
- Compare row counts and fingerprints where appropriate.

In [ ]:
control_record = {
    "checkpoint_after_publish": True,
    "idempotency_key": "Loan_ID",
    "bounded_replay": True,
    "deletion_strategy": "snapshot comparison",
    "chunk_size": chunk_size,
    "source_consistency_check": before_hash == after_hash,
}

assert all([
    control_record["checkpoint_after_publish"],
    control_record["bounded_replay"],
    control_record["source_consistency_check"],
    twice["Loan_ID"].is_unique,
])
print("DE-02 controls passed:", control_record)